In [10]:
import pandas as pd
import numpy as np
from datetime import timedelta
import re
from math import radians, sin, cos, sqrt, asin

# ==========================================
# 1. CONFIGURATION & DICTIONARIES
# ==========================================

# Map IATA Code -> Country Name
iata_country_map = {
    # Spain (Zone 0)
    'MAD': 'Spain', 'BCN': 'Spain', 'VLC': 'Spain', 'SVQ': 'Spain', 
    'AGP': 'Spain', 'BIO': 'Spain', 'ALC': 'Spain', 'PMI': 'Spain',
    
    # UK / Ireland / Portugal (Zone -1)
    'LON': 'UK', 'LHR': 'UK', 'LGW': 'UK', 'STN': 'UK', 'LTN': 'UK', 
    'LCY': 'UK', 'SEN': 'UK', 'EDI': 'UK', 'MAN': 'UK', 'BHX': 'UK',
    'DUB': 'Ireland',
    'LIS': 'Portugal', 'OPO': 'Portugal', 'FAO': 'Portugal',
    
    # Central Europe (Zone 0)
    'PAR': 'France', 'CDG': 'France', 'ORY': 'France', 'BVA': 'France', 'LYS': 'France', 'NCE': 'France',
    'ROM': 'Italy', 'FCO': 'Italy', 'CIA': 'Italy', 'MIL': 'Italy', 'MXP': 'Italy', 'LIN': 'Italy', 'BGY': 'Italy',
    'BER': 'Germany', 'FRA': 'Germany', 'MUC': 'Germany',
    'AMS': 'Netherlands', 'BRU': 'Belgium', 'CRL': 'Belgium',
    'VIE': 'Austria', 'ZRH': 'Switzerland', 'GVA': 'Switzerland',
    'PRG': 'Czech Republic', 'BUD': 'Hungary', 'BTS': 'Slovakia',
    'CPH': 'Denmark', 'OSL': 'Norway', 'STO': 'Sweden', 'ARN': 'Sweden', 'NYO': 'Sweden', 'VST': 'Sweden',
    
    # East Europe (Zone +1)
    'ATH': 'Greece', 'HEL': 'Finland', 'BUH': 'Romania', 'OTP': 'Romania', 'BBU': 'Romania',
    
    # Special (Turkey)
    'IST': 'Turkey', 'SAW': 'Turkey'
}

# Time Zone Offsets relative to Spain (MAD)
# 0 = Same time, -1 = 1 hour behind, 1 = 1 hour ahead
country_offsets_standard = {
    'UK': -1, 'Ireland': -1, 'Portugal': -1, 'Canary Islands': -1,
    'Greece': 1, 'Finland': 1, 'Romania': 1, 'Turkey': 1, # Turkey logic handled in function
    # Default is 0
}

# Coordinates for Distance Calculation
city_coords = {
    "Madrid": (40.4168, -3.7038), "London": (51.5074, -0.1278), "Paris": (48.8566, 2.3522),
    "Berlin": (52.5200, 13.4050), "Rome": (41.9028, 12.4964), "Bucharest": (44.4268, 26.1025),
    "Vienna": (48.2082, 16.3738), "Amsterdam": (52.3676, 4.9041), "Copenhagen": (55.6761, 12.5683),
    "Stockholm": (59.3293, 18.0686), "Oslo": (59.9139, 10.7522), "Helsinki": (60.1699, 24.9384),
    "Lisbon": (38.7223, -9.1393), "Istanbul": (41.0082, 28.9784), "Zurich": (47.3769, 8.5417),
    "Brussels": (50.8503, 4.3517), "Dublin": (53.3498, -6.2603), "Prague": (50.0755, 14.4378),
    "Budapest": (47.4979, 19.0402), "Athens": (37.9838, 23.7275), "Milan": (45.4642, 9.1900)
}

# Map IATA codes to the City Names used in 'city_coords'
iata_city_map = {
    'MAD': 'Madrid',
    'LON': 'London', 'LHR': 'London', 'LGW': 'London', 'STN': 'London', 'LTN': 'London', 'LCY': 'London', 'SEN': 'London',
    'PAR': 'Paris', 'CDG': 'Paris', 'ORY': 'Paris', 'BVA': 'Paris',
    'BER': 'Berlin',
    'ROM': 'Rome', 'FCO': 'Rome', 'CIA': 'Rome',
    'MIL': 'Milan', 'MXP': 'Milan', 'LIN': 'Milan', 'BGY': 'Milan',
    'BUH': 'Bucharest', 'OTP': 'Bucharest', 'BBU': 'Bucharest',
    'VIE': 'Vienna', 'AMS': 'Amsterdam', 'CPH': 'Copenhagen',
    'STO': 'Stockholm', 'ARN': 'Stockholm', 'NYO': 'Stockholm', 'VST': 'Stockholm',
    'OSL': 'Oslo', 'HEL': 'Helsinki', 'LIS': 'Lisbon',
    'IST': 'Istanbul', 'SAW': 'Istanbul', 'ZRH': 'Zurich',
    'BRU': 'Brussels', 'CRL': 'Brussels', 'DUB': 'Dublin',
    'PRG': 'Prague', 'BUD': 'Budapest', 'ATH': 'Athens'
}

# ==========================================
# 2. HELPER FUNCTIONS
# ==========================================

def parse_kayak_date(date_str):
    """Parse 'lun. 15/12' into datetime with correct year (2025/2026)."""
    try:
        match = re.search(r'(\d+)/(\d+)', str(date_str))
        if not match: return pd.NaT
        day, month = int(match.group(1)), int(match.group(2))
        year = 2025 if month == 12 else 2026
        return pd.Timestamp(year=year, month=month, day=day)
    except:
        return pd.NaT

def calculate_accurate_duration(row):
    """Calculate flight minutes adjusting for Time Zones and Turkey logic."""
    if pd.isnull(row['Arrival_Time']) or pd.isnull(row['Departure_Time']):
        return np.nan
        
    raw_minutes = (row['Arrival_Time'] - row['Departure_Time']).total_seconds() / 60
    
    offset_origin = country_offsets_standard.get(row['Origin_Country'], 0)
    offset_dest = country_offsets_standard.get(row['Dest_Country'], 0)
    
    # Turkey Logic: In winter, diff is +2h vs Spain (usually +1)
    is_winter = row['Date'].month in [11, 12, 1, 2, 3]
    if row['Origin_Country'] == 'Turkey' and is_winter: offset_origin = 2
    if row['Dest_Country'] == 'Turkey' and is_winter: offset_dest = 2
    
    adjustment = (offset_dest - offset_origin) * 60
    return raw_minutes - adjustment

def haversine(lat1, lon1, lat2, lon2):
    """Calculate distance in km between two lat/lon points."""
    R = 6371.0
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)
    a = sin(dlat/2)**2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon/2)**2
    c = 2 * asin(sqrt(a))
    return R * c

def compute_distance(row):
    """Get coords for Origin/Dest and calc distance."""
    city_origin = iata_city_map.get(row['origen_iata'])
    city_dest = iata_city_map.get(row['destino_iata'])
    
    if city_origin in city_coords and city_dest in city_coords:
        lat1, lon1 = city_coords[city_origin]
        lat2, lon2 = city_coords[city_dest]
        return haversine(lat1, lon1, lat2, lon2)
    return np.nan

# ==========================================
# 3. MAIN EXECUTION
# ==========================================

# Load Data (skip bad lines)
df1 = pd.read_csv('vuelos_kayak_data.csv', sep=';', engine='python', on_bad_lines='warn')
df2 = pd.read_csv('vuelos_kayak_data2.csv', sep=';', engine='python', on_bad_lines='warn')

# Merge them vertically
df = pd.concat([df1, df2], ignore_index=True)

# Remove exact duplicates (overlapping scrape data)
initial_len = len(df)
df.drop_duplicates(inplace=True)
print(f"Merged successfully. Total rows: {len(df)} (Removed {initial_len - len(df)} duplicates)")


# --- Date Parsing ---
df['Date'] = df['fecha_vuelo'].apply(parse_kayak_date)
df.dropna(subset=['Date'], inplace=True)

# --- Feature Engineering ---
df['Month'] = df['Date'].dt.month
df['Day_Name'] = df['Date'].dt.day_name()
df['Weekday'] = df['Date'].dt.weekday
df['Is_Weekend'] = df['Weekday'].isin([5, 6])

# --- Countries & Routes ---
df['Origin_Country'] = df['origen_iata'].map(iata_country_map).fillna('Unknown')
df['Dest_Country'] = df['destino_iata'].map(iata_country_map).fillna('Unknown')
df = df[
    (df["Origin_Country"].notna()) &
    (df["Dest_Country"].notna()) &
    (df["Origin_Country"] != "Unknown") &
    (df["Dest_Country"] != "Unknown")
].copy()
df['Route'] = df['origen_iata'] + '-' + df['destino_iata']

# --- Cities ---
df["Origin_City"] = df["origen_iata"].map(iata_city_map).fillna("Unknown")
df["Dest_City"] = df["destino_iata"].map(iata_city_map).fillna("Unknown")
df = df[
    (df["Origin_City"].notna()) &
    (df["Dest_City"].notna()) &
    (df["Origin_City"] != "Unknown") &
    (df["Dest_City"] != "Unknown")
].copy()


# --- Time Cleaning ---
df['hora_llegada_clean'] = df['hora_llegada'].astype(str).str.replace(r'["\n\+1]', '', regex=True).str.strip()

# Convert strings to Datetime (errors='coerce' handles bad times like ':55')
df['Departure_Time'] = pd.to_datetime(df['Date'].astype(str) + ' ' + df['hora_salida'], format='%Y-%m-%d %H:%M', errors='coerce')
df['Arrival_Time'] = pd.to_datetime(df['Date'].astype(str) + ' ' + df['hora_llegada_clean'], format='%Y-%m-%d %H:%M', errors='coerce')

# Remove rows with invalid times
df.dropna(subset=['Departure_Time', 'Arrival_Time'], inplace=True)

# Overnight adjustment (if arrival < departure, add 1 day)
mask_overnight = df['Arrival_Time'] < df['Departure_Time']
df.loc[mask_overnight, 'Arrival_Time'] += timedelta(days=1)
# Explicit +1 check
mask_explicit = df['hora_llegada'].astype(str).str.contains(r'\+1')
df.loc[mask_explicit & (~mask_overnight), 'Arrival_Time'] += timedelta(days=1)

# --- Calc Duration & Distance ---
df['Duration_Minutes'] = df.apply(calculate_accurate_duration, axis=1)
df['Distance_km'] = df.apply(compute_distance, axis=1)

# --- Airline Info ---
df['Primary_Airline'] = df['aerolinea'].str.split(',').str[0].str.strip()
df['Is_Mixed_Carrier'] = df['aerolinea'].str.contains(',') | (df['aerolinea'] == 'Varias aerolíneas')
df['Airline_Count'] = df['aerolinea'].str.count(',') + 1
df.loc[df['aerolinea'] == 'Varias aerolíneas', 'Airline_Count'] = 2 

# --- Stops ---
stop_mapping = {'directo': 0, '1 escala': 1, '2 escalas': 2, '2 stops': 2, '3 escalas': 3}
df['Stops_Numeric'] = df['paradas'].map(stop_mapping).fillna(0).astype(int)

# ==========================================
# 4. EXPORT
# ==========================================

final_cols = [
    'Date', 'Month', 'Day_Name', 'Weekday',
    'Departure_Time', 'Arrival_Time', 'Duration_Minutes', 'Distance_km',
    'precio', 'Stops_Numeric',
    'Primary_Airline', 'Is_Mixed_Carrier', 'Airline_Count',
    'origen_iata', 'destino_iata', 'Origin_Country', 'Dest_Country', 'Route'
]

df_clean = df[final_cols].copy()
df_clean.sort_values(by='Departure_Time', inplace=True)

print("Data processed successfully.")
print(df_clean[['Route', 'Duration_Minutes', 'Distance_km', 'precio']].head())

df_clean.to_csv('vuelos_kayak_clean_final.csv', index=False)

C:\Users\diego\AppData\Local\Temp\ipykernel_18180\3261320656.py:133: ParserWarning: Skipping line 32211: Expected 8 fields in line 32211, saw 15

  df1 = pd.read_csv('vuelos_kayak_data.csv', sep=';', engine='python', on_bad_lines='warn')


Merged successfully. Total rows: 90828 (Removed 4791 duplicates)
Data processed successfully.
         Route  Duration_Minutes  Distance_km  precio
10159  OTP-OSL             330.0  2004.341329   184.0
10162  OTP-OSL             330.0  2004.341329   150.0
85025  LIS-ZRH            1290.0  1722.895007   227.0
10166  OTP-OSL             315.0  2004.341329   300.0
11012  OTP-LIS            1460.0  2975.121245   240.0
